# Discovery, Validation, And Artifact Inspection

This notebook uses the shared Drive dataset as a read-only source, uses local Colab artifacts for reliability, reuses the training notebook runtime config when available, and exports the finished discovery/validation outputs to a clearly separate Drive folder.

In [ ]:
# Configuration — edit these values, then Run All
# This notebook reuses the saved training runtime config and only overrides the
# decode target for discovery.

DECODE_TYPE = 'stimulus_change'
DISCOVERY_DEVICE_MODE = 'auto'  # 'auto' = GPU encoder if available, 'cpu' = force CPU only
NOTEBOOK_STEP_LOG_EVERY = 16

In [ ]:
from google.colab import drive
from pathlib import Path
import shutil
import json
import pandas as pd
from datetime import datetime

drive.mount('/content/drive')
repo_root = Path('/content/predictive_circuit_coding')
github_repo_url = 'https://github.com/jacobposchl/predictive_circuit_coding'

# Shared folder from the storage-heavy Drive account: read dataset from here.
shared_drive_root = Path('/content/drive/MyDrive/pcc_runs')
drive_data_root = shared_drive_root / 'data' / 'allen_visual_behavior_neuropixels'

# Keep exported Colab outputs in a clearly different folder name so they do not get
# confused with the source dataset bundle.
drive_export_root = Path('/content/drive/MyDrive/pcc_colab_outputs')
drive_export_root.mkdir(parents=True, exist_ok=True)

print('Repo root:', repo_root)
print('GitHub repo:', github_repo_url)
print('Shared dataset root:', drive_data_root)
print('Drive export root:', drive_export_root)

In [ ]:
if not repo_root.exists():
    !git clone {github_repo_url} {repo_root}
%cd {repo_root}
# Keep Colab installs minimal to avoid restart prompts from preloaded widget packages.
!pip install -e .

In [ ]:
repo_dataset_root = repo_root / 'data' / 'allen_visual_behavior_neuropixels'
repo_artifacts_root = repo_root / 'artifacts'

assert drive_data_root.exists(), f'Missing Drive dataset bundle: {drive_data_root}'

repo_dataset_root.parent.mkdir(parents=True, exist_ok=True)
if repo_dataset_root.exists() or repo_dataset_root.is_symlink():
    if repo_dataset_root.is_symlink():
        repo_dataset_root.unlink()
    else:
        shutil.rmtree(repo_dataset_root)
repo_dataset_root.symlink_to(drive_data_root, target_is_directory=True)

if not repo_artifacts_root.exists():
    repo_artifacts_root.mkdir(parents=True, exist_ok=True)

print('Linked dataset into repo:', repo_dataset_root)
print('Using local artifact directory:', repo_artifacts_root)

In [ ]:
import yaml
from predictive_circuit_coding.utils import (
    NotebookStageReporter,
    build_notebook_discovery_runtime_config,
    describe_notebook_compute_targets,
    load_notebook_split_counts,
    resolve_notebook_checkpoint,
    restore_latest_exported_artifacts,
    run_notebook_discovery,
    run_notebook_validation,
    verify_paths_exist,
)

reporter = NotebookStageReporter(name='colab-discover', expected_duration='1-5 minutes for debug runs, longer for larger discovery passes')
reporter.banner('Discovery And Validation', subtitle='Frozen-token discovery, validation controls, inspection, and export')

data_config = repo_root / 'configs' / 'pcc' / 'allen_visual_behavior_neuropixels_local.yaml'
training_runtime_config = repo_root / 'colab_runtime_experiment.yaml'
runtime_experiment_config = repo_root / 'colab_discovery_runtime_experiment.yaml'
checkpoint_dir = repo_root / 'artifacts' / 'checkpoints'
summary_path = repo_root / 'artifacts' / 'training_summary.json'
restored_export_run = None

if not summary_path.exists() and not checkpoint_dir.exists():
    restored_export_run = restore_latest_exported_artifacts(
        drive_export_root=drive_export_root,
        local_artifact_root=repo_root / 'artifacts',
        runtime_experiment_config=training_runtime_config,
    )
elif not summary_path.exists() and checkpoint_dir.exists() and not any(checkpoint_dir.glob('*.pt')):
    restored_export_run = restore_latest_exported_artifacts(
        drive_export_root=drive_export_root,
        local_artifact_root=repo_root / 'artifacts',
        runtime_experiment_config=training_runtime_config,
    )

if not training_runtime_config.exists():
    raise FileNotFoundError(
        'Missing colab_runtime_experiment.yaml. Run the training notebook first, or restore a prior training export into artifacts/.'
    )

training_runtime_payload = yaml.safe_load(training_runtime_config.read_text(encoding='utf-8'))
experiment_config = build_notebook_discovery_runtime_config(
    source_experiment_config=training_runtime_config,
    runtime_experiment_config=runtime_experiment_config,
    artifact_root=repo_root / 'artifacts',
    decode_type=DECODE_TYPE,
    device_mode=DISCOVERY_DEVICE_MODE,
    step_log_every=NOTEBOOK_STEP_LOG_EVERY,
)
compute_targets = describe_notebook_compute_targets(experiment_config_path=experiment_config)
runtime_subset_payload = training_runtime_payload.get('runtime_subset') or {}
split_manifest_path = Path(runtime_subset_payload.get('split_manifest_path', repo_root / 'data' / 'allen_visual_behavior_neuropixels' / 'splits' / 'split_manifest.json'))
split_counts = load_notebook_split_counts(
    split_manifest_path=split_manifest_path,
)

selected_checkpoint = resolve_notebook_checkpoint(summary_path=summary_path, checkpoint_dir=checkpoint_dir)
run_label = datetime.now().strftime('discover_run_%Y%m%d_%H%M%S')
drive_run_export_root = drive_export_root / run_label

print(f'Training runtime config: {training_runtime_config}')
print(f'Discovery runtime config: {experiment_config}')
print(f'Decode type: {DECODE_TYPE}')
print(
    f"Device mode: {DISCOVERY_DEVICE_MODE} | "
    f"encoder={compute_targets['encoder_device']} | "
    f"probe={compute_targets['probe_device']} | "
    f"clustering={compute_targets['clustering_device']} | "
    f"metrics={compute_targets['metrics_device']}"
)
print(f'Runtime split manifest: {split_manifest_path} | checkpoint: {selected_checkpoint.name}')
print(f'Realized split counts: {split_counts}')
if restored_export_run is not None:
    print(f'Restored exported training run from: {restored_export_run}')

In [ ]:
paths_ok = verify_paths_exist({
    'experiment_config': experiment_config,
    'data_config': data_config,
    'checkpoint': selected_checkpoint,
    'drive_dataset_root': drive_data_root,
})
print(paths_ok)
assert all(paths_ok.values()), 'Missing discovery/validation inputs.'

print('Using runtime subset described by the training notebook artifacts.')
print('Realized split counts:', split_counts)

In [ ]:
reporter.begin('discovery', next_artifact='decode coverage summary + discovery artifact + cluster summary')
%cd {repo_root}
discovery_run = run_notebook_discovery(
    experiment_config_path=experiment_config,
    data_config_path=data_config,
    checkpoint_path=selected_checkpoint,
    split_name='discovery',
    progress_ui=True,
)
print('Discovery artifact', discovery_run.discovery_artifact_path)
print('Decode coverage summary', discovery_run.decode_coverage_summary_path)
print('Cluster summary', discovery_run.cluster_summary_json_path)
reporter.finish('discovery')

In [ ]:
discovery_artifact = discovery_run.discovery_artifact_path
decode_coverage_json = discovery_run.decode_coverage_summary_path
cluster_summary_json = discovery_run.cluster_summary_json_path
cluster_summary_csv = discovery_run.cluster_summary_csv_path

reporter.begin('validation', next_artifact='local validation summary json/csv')
%cd {repo_root}
validation_run = run_notebook_validation(
    experiment_config_path=experiment_config,
    data_config_path=data_config,
    checkpoint_path=selected_checkpoint,
    discovery_artifact_path=discovery_artifact,
    progress_ui=True,
)
print('Validation summary', validation_run.validation_summary_json_path)
reporter.finish('validation')

## Results

Visualization and summary of the completed run.


In [ ]:
import json
import pandas as pd

# Define validation artifact paths (produced by the validation step above).
validation_json = validation_run.validation_summary_json_path
validation_csv  = validation_run.validation_summary_csv_path
with open(decode_coverage_json, 'r', encoding='utf-8') as fh:
    decode_coverage_payload = json.load(fh)
print('=== Decode Coverage Summary ===')
print(
    f"  split={decode_coverage_payload['split_name']}  "
    f"target={decode_coverage_payload['target_label']}  "
    f"scanned={decode_coverage_payload['total_scanned_windows']}  "
    f"positive={decode_coverage_payload['positive_window_count']}  "
    f"negative={decode_coverage_payload['negative_window_count']}  "
    f"selected_positive={decode_coverage_payload['selected_positive_count']}  "
    f"selected_negative={decode_coverage_payload['selected_negative_count']}"
)
print(f"  positive-window sessions: {decode_coverage_payload['sessions_with_positive_windows']}")


# --- Cluster summary table ---
print('=== Cluster Summary ===')
cluster_df = pd.read_csv(cluster_summary_csv)
display(cluster_df)

# --- Per-cluster one-liners ---
with open(cluster_summary_json, 'r', encoding='utf-8') as fh:
    cluster_summary_payload = json.load(fh)

print(
    f"\nTotal clusters: {cluster_summary_payload['cluster_count']}  "
    f"Total candidates: {cluster_summary_payload['candidate_count']}"
)
for cluster in cluster_summary_payload.get('clusters', []):
    top_regions = ', '.join(r['value'] for r in cluster.get('top_regions', [])[:3]) or 'n/a'
    print(
        f"  Cluster {cluster['cluster_id']:>3}: "
        f"{cluster['candidate_count']} candidates, "
        f"{cluster['session_count']} sessions, "
        f"mean score {cluster['mean_score']:.3f}, "
        f"top regions: {top_regions}"
    )

# --- Validation summary ---
if validation_json.exists():
    print('\n=== Validation Summary ===')
    with open(validation_json, 'r', encoding='utf-8') as fh:
        validation_payload = json.load(fh)
    real_acc  = validation_payload.get('real_label_metrics', {}).get('probe_accuracy', float('nan'))
    shuf_acc  = validation_payload.get('shuffled_label_metrics', {}).get('probe_accuracy', float('nan'))
    test_acc  = validation_payload.get('held_out_test_metrics', {}).get('probe_accuracy', float('nan'))
    held_out_auc = validation_payload.get('held_out_similarity_summary', {}).get('window_roc_auc', float('nan'))
    held_out_pr = validation_payload.get('held_out_similarity_summary', {}).get('window_pr_auc', float('nan'))
    silhouette = validation_payload.get('cluster_quality_summary', {}).get('silhouette_score', float('nan'))
    persistence = validation_payload.get('cluster_quality_summary', {}).get('cluster_persistence_mean', float('nan'))
    print(f'  Probe accuracy:  discovery={real_acc:.3f}  shuffled={shuf_acc:.3f}')
    print(f'  Held-out test accuracy: {test_acc:.3f}')
    print(f'  Held-out motif similarity: ROC-AUC={held_out_auc:.3f}  PR-AUC={held_out_pr:.3f}')
    print(f'  Cluster quality: silhouette={silhouette:.3f}  persistence_mean={persistence:.3f}')
    issues = validation_payload.get('provenance_issues', [])
    if issues:
        print(f'  Provenance issues ({len(issues)}):')
        for issue in issues:
            print(f'    - {issue}')
    display(pd.read_csv(validation_csv))
else:
    print('\nValidation artifacts not found — run the validation cell above first.')

In [ ]:
import json
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from collections import Counter

with open(cluster_summary_json, 'r', encoding='utf-8') as fh:
    cluster_payload = json.load(fh)
clusters = cluster_payload.get('clusters', [])

val_payload = None
if validation_json.exists():
    with open(validation_json, 'r', encoding='utf-8') as fh:
        val_payload = json.load(fh)

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

# Panel 1: Candidate counts per cluster
cluster_ids = [str(c['cluster_id']) for c in clusters]
cand_counts = [c['candidate_count'] for c in clusters]
axes[0].bar(cluster_ids, cand_counts, color='#4c72b0')
axes[0].set_title('Candidates per Cluster')
axes[0].set_xlabel('Cluster ID')
axes[0].set_ylabel('Candidate Count')
axes[0].xaxis.set_major_locator(mticker.MaxNLocator(integer=True, nbins=10))
for i, v in enumerate(cand_counts):
    axes[0].text(i, v + 0.3, str(v), ha='center', va='bottom', fontsize=8)

# Panel 2: Discovery/test probe accuracy and held-out motif similarity
if val_payload is not None:
    real_acc = val_payload.get('real_label_metrics', {}).get('probe_accuracy', float('nan'))
    shuf_acc = val_payload.get('shuffled_label_metrics', {}).get('probe_accuracy', float('nan'))
    test_acc = val_payload.get('held_out_test_metrics', {}).get('probe_accuracy', float('nan'))
    held_out_auc = val_payload.get('held_out_similarity_summary', {}).get('window_roc_auc', float('nan'))
    held_out_pr = val_payload.get('held_out_similarity_summary', {}).get('window_pr_auc', float('nan'))
    bars = axes[1].bar(['Discovery', 'Shuffled', 'Held-out test', 'Motif ROC-AUC', 'Motif PR-AUC'], [real_acc, shuf_acc, test_acc, held_out_auc, held_out_pr],
                       color=['#55a868', '#dd8452', '#4c72b0', '#c44e52', '#64b5cd'], width=0.55)
    axes[1].set_title('Held-out Probe And Motif Metrics')
    axes[1].set_ylabel('Accuracy')
    axes[1].set_ylim(0, 1.1)
    axes[1].axhline(0.5, color='gray', linestyle='--', linewidth=0.8, label='Chance')
    axes[1].legend(fontsize=8)
    for bar, v in zip(bars, [real_acc, shuf_acc, test_acc, held_out_auc, held_out_pr]):
        if v == v:
            axes[1].text(bar.get_x() + bar.get_width() / 2, v + 0.03, f'{v:.3f}', ha='center', fontsize=9)
else:
    axes[1].text(0.5, 0.5, 'Validation not run', ha='center', va='center',
                 transform=axes[1].transAxes, fontsize=11, color='gray')
    axes[1].set_title('Held-out Probe And Motif Metrics')
    axes[1].axis('off')

# Panel 3: Top brain regions across all clusters
region_counter = Counter()
for cluster in clusters:
    for entry in cluster.get('top_regions', []):
        region_counter[entry['value']] += entry['count']

if region_counter:
    top_regions = region_counter.most_common(10)
    names  = [r[0] for r in reversed(top_regions)]
    counts = [r[1] for r in reversed(top_regions)]
    axes[2].barh(names, counts, color='#8172b2')
    axes[2].set_title('Top Brain Regions (all clusters)')
    axes[2].set_xlabel('Cumulative Candidate Count')
    for i, v in enumerate(counts):
        axes[2].text(v + 0.1, i, str(v), va='center', fontsize=8)
else:
    axes[2].text(0.5, 0.5, 'No region data', ha='center', va='center',
                 transform=axes[2].transAxes, fontsize=11, color='gray')
    axes[2].axis('off')

fig.suptitle('Discovery & Validation Results', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
import json

with open(cluster_summary_json, 'r', encoding='utf-8') as fh:
    cluster_payload = json.load(fh)

cluster_count  = cluster_payload['cluster_count']
candidate_count = cluster_payload['candidate_count']

val_payload = None
if validation_json.exists():
    with open(validation_json, 'r', encoding='utf-8') as fh:
        val_payload = json.load(fh)

if val_payload is not None:
    real_acc   = val_payload.get('real_label_metrics', {}).get('probe_accuracy')
    shuf_acc   = val_payload.get('shuffled_label_metrics', {}).get('probe_accuracy')
    test_acc   = val_payload.get('held_out_test_metrics', {}).get('probe_accuracy')
    held_out_auc = val_payload.get('held_out_similarity_summary', {}).get('window_roc_auc')
    held_out_pr = val_payload.get('held_out_similarity_summary', {}).get('window_pr_auc')
    silhouette = val_payload.get('cluster_quality_summary', {}).get('silhouette_score')
    persistence = val_payload.get('cluster_quality_summary', {}).get('cluster_persistence_mean')

    acc_str  = f'{real_acc * 100:.1f}%'  if real_acc   is not None else 'n/a'
    shuf_str = f'{shuf_acc * 100:.1f}%'  if shuf_acc   is not None else 'n/a'
    test_str = f'{test_acc * 100:.1f}%'   if test_acc   is not None else 'n/a'
    auc_str = f'{held_out_auc:.3f}'       if held_out_auc is not None else 'n/a'
    pr_str = f'{held_out_pr:.3f}'         if held_out_pr is not None else 'n/a'
    sil_str = f'{silhouette:.2f}'         if silhouette is not None else 'n/a'
    persist_str = f'{persistence:.2f}'    if persistence is not None else 'n/a'

    is_structured = real_acc and shuf_acc and real_acc > shuf_acc + 0.05
    structure_word = 'structured' if is_structured else 'marginal or no'

    print(
        f'Discovery found {cluster_count} cluster(s) containing {candidate_count} candidate neural windows.\n'
        f'A linear probe trained on the real cluster labels achieved {acc_str} accuracy, '
        f'compared to {shuf_str} on shuffled labels, '
        f'and transferred to held-out test windows at {test_str}. '
        f'This suggests the clusters capture {structure_word} label-related structure.\n'
        f'Held-out motif similarity reached ROC-AUC {auc_str} and PR-AUC {pr_str}, '
        f'with silhouette {sil_str} and mean cluster persistence {persist_str}.'
    )
else:
    print(
        f'Discovery found {cluster_count} cluster(s) containing {candidate_count} candidate neural windows.\n'
        f'Validation was not run — probe transfer, held-out motif similarity, and cluster-quality metrics are not available.'
    )

In [ ]:
reporter.begin('export', next_artifact='Drive copy of local artifacts')
if drive_run_export_root.exists():
    shutil.rmtree(drive_run_export_root)
shutil.copytree(repo_artifacts_root, drive_run_export_root)
print('Exported local artifacts to:', drive_run_export_root)
reporter.finish('export')